# Analisis Komparasi Kinerja Algoritma Decision Tree C4.5 dan Random Forest
# untuk Klasifikasi Status Stunting pada Balita

**Skripsi**

---

## Daftar Isi
1. [Import Libraries](#1-import-libraries)
2. [Load Dataset](#2-load-dataset)
3. [Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis)
4. [Data Preprocessing](#4-data-preprocessing)
5. [Model Training](#5-model-training)
6. [Model Evaluation](#6-model-evaluation)
7. [Model Comparison](#7-model-comparison)
8. [Conclusion](#8-conclusion)

## 1. Import Libraries

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve, auc
)

# System
import sys
from pathlib import Path

# Add parent directory to path
sys.path.append('..')

# Project modules
from src.config import *
from src.preprocessing import DataPreprocessor
from src.models import ModelTrainer, compare_models
from src.visualization import Visualizer

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 2. Load Dataset

In [ ]:
# Download dataset if not exists
if not RAW_DATA_FILE.exists():
    import kagglehub
    import shutil
    import os
    
    path = kagglehub.dataset_download(DATASET_NAME)
    for file in os.listdir(path):
        if file.endswith('.csv'):
            shutil.copy(os.path.join(path, file), RAW_DATA_FILE)
            print(f"Dataset downloaded to: {RAW_DATA_FILE}")

# Load dataset
df = pd.read_csv(RAW_DATA_FILE)
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
# Display first few rows
df.head(10)

In [ ]:
# Dataset info
df.info()

In [ ]:
# Column names
print("Columns:")
for col in df.columns:
    print(f"  - {col}")

## 3. Exploratory Data Analysis (EDA)

### 3.1 Basic Statistics

In [ ]:
# Descriptive statistics
df.describe()

In [ ]:
# Check missing values
print("Missing Values:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

In [ ]:
# Check duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")

### 3.2 Target Variable Distribution

In [ ]:
# Target distribution
target_counts = df[TARGET_COLUMN].value_counts()
print("Target Distribution:")
print(target_counts)
print("\nPercentage:")
print((target_counts / len(df) * 100).round(2))

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = sns.color_palette("husl", len(target_counts))
bars = axes[0].bar(target_counts.index, target_counts.values, color=colors, edgecolor='black')
axes[0].set_xlabel('Status Gizi', fontsize=12)
axes[0].set_ylabel('Jumlah', fontsize=12)
axes[0].set_title('Distribusi Status Gizi Balita', fontsize=14, fontweight='bold')

for bar, val in zip(bars, target_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                f'{val:,}', ha='center', va='bottom', fontsize=10)

# Pie chart
axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
           colors=colors, explode=[0.02]*len(target_counts), shadow=True)
axes[1].set_title('Proporsi Status Gizi Balita', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.3 Numerical Features Distribution

In [ ]:
# Distribution of numerical features
num_cols = [col for col in NUMERICAL_FEATURES if col in df.columns]

fig, axes = plt.subplots(2, len(num_cols), figsize=(7*len(num_cols), 10))

for i, col in enumerate(num_cols):
    # Histogram
    axes[0, i].hist(df[col], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, i].set_xlabel(col, fontsize=11)
    axes[0, i].set_ylabel('Frekuensi', fontsize=11)
    axes[0, i].set_title(f'Distribusi {col}', fontsize=12, fontweight='bold')
    
    # Add mean and median lines
    mean_val = df[col].mean()
    median_val = df[col].median()
    axes[0, i].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.2f}')
    axes[0, i].axvline(median_val, color='green', linestyle='--', label=f'Median: {median_val:.2f}')
    axes[0, i].legend()
    
    # Box plot
    sns.boxplot(data=df, y=col, ax=axes[1, i], color='lightblue')
    axes[1, i].set_title(f'Box Plot {col}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_PATH / 'numerical_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

### 3.4 Feature Distribution by Target

In [ ]:
# Feature distributions by target class
for col in num_cols:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    categories = df[TARGET_COLUMN].unique()
    colors = sns.color_palette("husl", len(categories))
    
    for cat, color in zip(categories, colors):
        subset = df[df[TARGET_COLUMN] == cat][col]
        ax.hist(subset, bins=40, alpha=0.5, label=cat, color=color, edgecolor='black')
    
    ax.set_xlabel(col, fontsize=12)
    ax.set_ylabel('Frekuensi', fontsize=12)
    ax.set_title(f'Distribusi {col} per Status Gizi', fontsize=14, fontweight='bold')
    ax.legend(title='Status Gizi')
    
    plt.tight_layout()
    plt.show()

### 3.5 Categorical Features

In [ ]:
# Categorical features distribution
cat_cols = [col for col in CATEGORICAL_FEATURES if col in df.columns]

for col in cat_cols:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    cross_tab = pd.crosstab(df[col], df[TARGET_COLUMN])
    cross_tab.plot(kind='bar', ax=ax, colormap='husl', edgecolor='black')
    
    ax.set_xlabel(col, fontsize=12)
    ax.set_ylabel('Jumlah', fontsize=12)
    ax.set_title(f'Distribusi {col} berdasarkan Status Gizi', fontsize=14, fontweight='bold')
    ax.legend(title='Status Gizi')
    ax.tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.savefig(FIGURES_PATH / f'categorical_{col.replace(" ", "_").lower()}.png', dpi=300, bbox_inches='tight')
    plt.show()

## 4. Data Preprocessing

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor()

# Run full preprocessing pipeline
data = preprocessor.full_pipeline(RAW_DATA_FILE)

In [ ]:
# Extract data
X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
feature_names = data['feature_names']
class_names = list(data['target_encoder'].classes_)

print(f"\nTraining set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")
print(f"Features: {feature_names}")
print(f"Classes: {class_names}")

## 5. Model Training

In [ ]:
# Initialize trainer
trainer = ModelTrainer()

### 5.1 Decision Tree C4.5

In [ ]:
# Create Decision Tree model (C4.5 uses entropy/information gain)
dt_model = trainer.create_decision_tree()

print("Decision Tree Parameters:")
print(dt_model.get_params())

In [ ]:
# Hyperparameter tuning for Decision Tree
print("Hyperparameter Tuning for Decision Tree C4.5...")
dt_model, dt_best_params = trainer.hyperparameter_tuning(
    dt_model, DT_PARAM_GRID, X_train, y_train, "Decision Tree C4.5"
)

In [ ]:
# Cross-validation
dt_cv_results = trainer.cross_validate(dt_model, X_train, y_train, "Decision Tree C4.5", cv=CV_FOLDS)

In [ ]:
# Train final model
dt_model = trainer.train_model(dt_model, X_train, y_train, "Decision Tree C4.5")

### 5.2 Random Forest

In [ ]:
# Create Random Forest model
rf_model = trainer.create_random_forest()

print("Random Forest Parameters:")
print(rf_model.get_params())

In [ ]:
# Hyperparameter tuning for Random Forest
print("Hyperparameter Tuning for Random Forest...")
rf_model, rf_best_params = trainer.hyperparameter_tuning(
    rf_model, RF_PARAM_GRID, X_train, y_train, "Random Forest"
)

In [ ]:
# Cross-validation
rf_cv_results = trainer.cross_validate(rf_model, X_train, y_train, "Random Forest", cv=CV_FOLDS)

In [ ]:
# Train final model
rf_model = trainer.train_model(rf_model, X_train, y_train, "Random Forest")

## 6. Model Evaluation

### 6.1 Decision Tree Evaluation

In [ ]:
# Evaluate Decision Tree
dt_results = trainer.evaluate_model(dt_model, X_test, y_test, "Decision Tree C4.5", class_names)

In [ ]:
# Feature importance for Decision Tree
dt_importance = trainer.get_feature_importance(dt_model, feature_names, "Decision Tree C4.5")

### 6.2 Random Forest Evaluation

In [ ]:
# Evaluate Random Forest
rf_results = trainer.evaluate_model(rf_model, X_test, y_test, "Random Forest", class_names)

In [ ]:
# Feature importance for Random Forest
rf_importance = trainer.get_feature_importance(rf_model, feature_names, "Random Forest")

## 7. Model Comparison

In [ ]:
# Collect results
results = {
    "Decision Tree C4.5": dt_results,
    "Random Forest": rf_results
}

# Compare models
comparison_df = compare_models(results)

In [ ]:
# Visualize comparison
visualizer = Visualizer()

# Metrics comparison bar chart
visualizer.plot_metrics_comparison(results)

In [ ]:
# Confusion matrices
for model_name, result in results.items():
    visualizer.plot_confusion_matrix(y_test, result['y_pred'], class_names, model_name)

In [ ]:
# ROC curves
visualizer.plot_roc_curves(results, y_test, class_names)

In [ ]:
# Feature importance comparison
visualizer.plot_feature_importance(dt_importance, "Decision Tree C4.5")
visualizer.plot_feature_importance(rf_importance, "Random Forest")

In [ ]:
# Decision Tree visualization
visualizer.plot_decision_tree(dt_model, feature_names, class_names, "Decision Tree C4.5")

In [ ]:
# Cross-validation comparison
visualizer.plot_cv_results(trainer.cv_results)

## 8. Save Models and Results

In [ ]:
# Save models
trainer.save_model(dt_model, "Decision Tree C4.5")
trainer.save_model(rf_model, "Random Forest")

In [ ]:
# Save results to CSV
comparison_data = []
for model_name, result in results.items():
    comparison_data.append({
        'Model': model_name,
        'Accuracy': result['accuracy'],
        'Precision': result['precision'],
        'Recall': result['recall'],
        'F1-Score': result['f1'],
        'ROC-AUC': result['roc_auc'],
        'Training Time (s)': result['training_time']
    })

results_df = pd.DataFrame(comparison_data)
results_df.to_csv(REPORTS_PATH / 'model_comparison.csv', index=False)
print("Results saved to reports/model_comparison.csv")

results_df

## 9. Conclusion

### Summary of Findings:

Based on the comparative analysis between Decision Tree C4.5 and Random Forest algorithms for stunting classification:

1. **Accuracy**: [To be filled based on results]
2. **Precision**: [To be filled based on results]
3. **Recall**: [To be filled based on results]
4. **F1-Score**: [To be filled based on results]
5. **ROC-AUC**: [To be filled based on results]

### Recommendations:

[To be filled based on results]